Let's formally test and develop a differentiable SAS model over here.

In [21]:
from functools import partial

import pandas as pd
import jax
import jax.numpy as jnp

import equinox as eqx
from equinox.nn import MLP, Identity, Linear

from typing import Callable
from jaxtyping import Array

import matplotlib.pyplot as plt


# Parameter and coefficients

In [138]:
# Number of time step
nt = 5

# Time duration
t_length = 2.

# Time step
dt = t_length / nt

# The maximum age step
τ_max = 3

# The old-water concentration
C_old = jnp.array([0.0]) # [mg/l]

# The first-order reaction rate constant
k1 = jnp.array([0.0])

# The equilibrium concentration
C_eq = jnp.array([0.0]) # [mg/o]

# Fractionation
α_Q, α_ET = 1., 0.


# Flow and solutes in/out

In [14]:
pulse_start1 = 0.05
pulse_end1 = 0.45

pulse_start2 = 1.25
pulse_end2 = 1.45

C_tracer_input1 = 0.5
C_tracer_input2 = 1.5
Q_steady = 1.
Storage_vol = 0.1

data_df = pd.DataFrame(index=jnp.arange(nt) * dt)
data_df['Q'] = Q_steady
data_df['ET'] = 0.
data_df['J'] = Q_steady
data_df['C_J'] = 0
data_df.loc[pulse_start1:pulse_end1, 'C_J'] = C_tracer_input1
data_df.loc[pulse_start1:pulse_end1, 'J'] = Q_steady * 2
data_df.loc[pulse_start1:pulse_end1, 'Q'] = Q_steady / 20.


/tmp/ipykernel_2299889/526309421.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data_df.loc[pulse_start1:pulse_end1, 'C_J'] = C_tracer_input1


# SAS functions

In [28]:
# The base class
class SAS(eqx.Module):
    loc: Array
    scale: Array
    
    def __init__(self, loc, scale):
        self.loc = loc
        self.scale = scale
    
    def pdf(self, Si, x):
        # Return PDF
        raise Exception('Not implemented')
    
    def __call__(self, Si, x):
        # Return CDF:
        raise Exception('Not implemented')


In [277]:
# The base class
class SAS_null(SAS):
    loc: Array
    scale: Array
    
    def pdf(self, Si, x):
        # Return PDF
        return 0.
    
    def __call__(self, Si, x):
        # Return CDF:
        return 0.


In [30]:
# Gamme distribution
class SAS_Gamma(SAS):
    a: Array
    
    def __init__(self, a, loc, scale):
        super().__init__(loc, scale)
        self.a = a
    
    def pdf(self, Si, x=None):
        a, loc, scale = self.a, self.loc, self.scale
        return jax.scipy.stats.gamma.pdf(Si, a, loc, scale)
    
    def __call__(self, Si, x=None):
        a, loc, scale = self.a, self.loc, self.scale
        return jax.scipy.stats.gamma.cdf(Si, a, loc, scale)


In [55]:
# Beta distribution
class SAS_Beta(SAS):
    a: Array
    b: Array
    
    def __init__(self, a, b, loc, scale):
        super().__init__(loc, scale)
        self.a, self.b = a, b
    
    def pdf(self, Si, x=None):
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        return jax.scipy.stats.beta.pdf(Si, a, b, loc, scale)
    
    def __call__(self, Si, x=None):
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        return jax.scipy.stats.beta.cdf(Si, a, b, loc, scale)


In [270]:
# Kumaraswamy distribution 
class SAS_Kumaraswamy(SAS):
    a: Array
    b: Array
    
    def __init__(self, a, b, loc, scale):
        super().__init__(loc, scale)
        self.a, self.b = a, b
    
    def pdf(self, Si, x=None):
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        y = (Si - loc) / scale
        y = jax.lax.min(jax.lax.max(0., y), 1.)
        return a * b * y ** (a-1) * (1-y**a) ** (b-1)
    
    def __call__(self, Si, x=None):
        # Return CDF:
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        y = (Si - loc) / scale
        y = jax.lax.min(jax.lax.max(0., y), 1.)
        return 1. - (1. - y**a) ** b


In [104]:
# Mixture density network
class SAS_MDN(SAS):
    m: eqx.Module
    m_α: eqx.Module
    m_μ: eqx.Module
    m_σ: eqx.Module
    pdf_func: Callable
    cdf_func: Callable
    
    def __init__(self, loc, scale, n_input, n_hidden, n_mixture, key, **mlp_kwargs):
        super().__init__(loc, scale)
        key1, key2, key3, key4 = jax.random.split(key,4)
        
        # MLP model for predicting the hidden states
        # n_mlp_output = (n_output+2) * n_mixture
        self.m = MLP(in_size=n_input, out_size=n_hidden, key=key1, **mlp_kwargs)
        
        # MLP model for predicting α, μ, and σ
        self.m_α = Linear(in_features=n_hidden, out_features=n_mixture, key=key2)
        self.m_μ = Linear(in_features=n_hidden, out_features=n_mixture, key=key3)
        self.m_σ = Linear(in_features=n_hidden, out_features=n_mixture, key=key4)
        
        # PDF function of mixture distributions 
        self.pdf_func = jax.scipy.stats.norm.pdf
        
        # CDF function of mixture distributions 
        self.cdf_func = jax.scipy.stats.norm.cdf
    
    def get_param(self, x):
        # Calculate the weights, mean, and standard deviation of the mixture distributions
        z = self.m(x)  # shape: (n_hidden,)
        z_α, z_μ, z_σ = self.m_α(z), self.m_μ(z), self.m_σ(z) # shape: (n_mixture,) (n_mixture,) (n_mixture)
        
        # The weights (n_mixture,)
        α = jax.nn.softmax(z_α)
        
        # The mean (n_mixture,)
        μ = z_μ
        
        # The standard deviation (n_mixture,)
        σ = jnp.exp(z_σ)
        
        return α, μ, σ
    
    def pdf(self, Si, x):
        y = (Si - self.loc) / self.scale
        α, μ, σ = self.get_param(x)
        pdfs = jax.vmap(self.pdf_func, in_axes=(None,0,0))(y, μ, σ)
        return jnp.sum(jnp.dot(α, pdfs))
    
    def __call__(self, Si, x):
        y = (Si - self.loc) / self.scale
        α, μ, σ = self.get_param(x)
        cdfs = jax.vmap(self.cdf_func, in_axes=(None,0,0))(y, μ, σ)
        return jnp.sum(jnp.dot(α, cdfs))


# SAS-based transport forward function

In [15]:
# Compute mQ -- mass loss with flow
def compute_mQ(sT, mT, α, Q_t, pQ):
    # sT, mT, α, Q_t, pQ: (1,), (1,), (1,), (1,), (1,)
    # return mT / sT * (α * Q_t * pQ)
    # return jnp.where(sT == 0.0, 0.0, mT / sT * (α * Q_t * pQ))
    part_a = mT * (α * Q_t * pQ)
    y = jnp.where(sT == 0.0, 1.0, sT)
    part_b = jnp.where(sT == 0.0, 0.0, 1./ y)
    return part_a * part_b

# Compute mR -- Reaction rate
def compute_mR(sT, mT, k1, C_eq):
    # sT, mT, k1, C_eq: (1,), (1,), (1,), (1,)
    return k1 * (C_eq*sT - mT)

compute_mR_vec = jax.vmap(compute_mR, in_axes=(None,0,0,0))


In [60]:
def f(sTmT, ST, dt, 
      J_τ_t, C_J_τ_t, Q_t, ET_t, 
      sas_Q, sas_ET, sas_arg_Q, sas_arg_ET, 
      α_Q, α_ET, k1, C_eq
):
    # sTmT: (1+nm,)
    # ST: (1,)
    # dt: (1,)
    # J_τ_t: (1,)
    # C_J_τ_t: (nm,)
    # Q_t: (1,)
    # ET_t: (1,)
    # α_Q: (1,)
    # α_ET: (1,)
    # sas_Q: (1,)
    # sas_arg_Q: (1,)
    # sas_ET: (1,)
    # sas_arg_ET: (1,)
    # k1: (nm,)
    # C_eq: (nm,)
    
    sT = sTmT[0] # (1,)
    mT = sTmT[1:] # (nm,)
    
    # Calculate the pQ, mQ, and mR
    PQ1, PQ2 = sas_Q(ST+sT*dt, sas_arg_Q), sas_Q(ST, sas_arg_Q)
    pQ = (PQ1 - PQ2) / dt
    PET1, PET2 = sas_ET(ST+sT*dt, sas_arg_ET), sas_ET(ST, sas_arg_ET)
    pET = (PET1 - PET2) / dt
    mQ = compute_mQ(sT, mT, α_Q, Q_t, pQ) # (nm,)
    mET = compute_mQ(sT, mT, α_ET, ET_t, pET) # (nm,)
    
    # Calculate mR
    mR = compute_mR_vec(sT, mT, k1, C_eq)  # (nm,)
    
    # Update the change of sT and mT
    δs = (J_τ_t - Q_t * pQ * dt - ET_t * pET * dt) / dt  # (1,)

    # Update the change of sT and mT
    δm = (J_τ_t * C_J_τ_t - mQ * dt - mET * dt + mR * dt) / dt  # (nm,)
    return jnp.concat([jnp.array([δs]), δm])

fvec = jax.vmap(
    f, in_axes=(0,0,None, 0,0,0,0,None,None,0,0,
                None,None,None,None)
)


# A general solver to solve TTDs-like function

In [192]:
# !!!! Note that the usage of nn.relu would affect the accuracy of implicit differentiation
def solve_step_euler(f, states, ST_top, ST_bot, dt, *args):
    # states: (nt, ns)
    # ST_top, ST_bot: (nt)

    # Note: we use relu to make sure the updated states are non-zero
    ST = ST_top
    r = f(states, ST, dt, *args)

    states_new = states + r * dt
    # states_new = jax.nn.relu(states + (r1 + r2)/2. * dt)

    return states_new


In [33]:
# !!!! Note that the usage of nn.relu would affect the accuracy of implicit differentiation
def solve_step_rk2(f, states, ST_top, ST_bot, dt, *args):
    # states: (nt, ns)
    # ST_top, ST_bot: (nt)

    # Note: we use relu to make sure the updated states are non-zero
    ST1 = ST_top
    states1 = states
    r1 = f(states1, ST1, dt, *args)
    
    ST2 = ST_bot
    # states2 = jax.nn.relu(states+1.*r1*dt)
    states2 = states+1.*r1*dt
    r2 = f(states2, ST2, dt, *args)

    states_new = states + (r1 + r2)/2. * dt
    # states_new = jax.nn.relu(states + (r1 + r2)/2. * dt)

    return states_new


In [77]:
# !!!! Note that the usage of nn.relu would affect the accuracy of implicit differentiation
def solve_step_rk4(f, states, ST_top, ST_bot, dt, *args):
    # states: (nt, ns)
    # ST_top, ST_bot: (nt)

    # Note: we use relu to make sure the updated states are non-zero
    ST1 = ST_top
    states1 = states
    r1 = f(states1, ST1, dt, *args)
    
    ST2 = ST_top/2 + ST_bot/2
    states2 = jax.nn.relu(states+0.5*r1*dt)
    # states2 = states+0.5*r1*dt
    r2 = f(states2, ST2, dt, *args)
    
    ST3 = ST_top/2 + ST_bot/2
    states3 = jax.nn.relu(states+0.5*r2*dt)
    # states3 = states+0.5*r2*dt
    r3 = f(states3, ST3, dt, *args)
    
    ST4 = ST_bot
    states4 = jax.nn.relu(states+1.*r3*dt)
    # states4 = states+1.*r3*dt
    r4 = f(states4, ST4, dt, *args)

    # states_new = states + 1./6 * (r1 + 2 * r2 + 2 * r3 + r4) * dt
    states_new = jax.nn.relu(states + 1./6 * (r1 + 2 * r2 + 2 * r3 + r4) * dt)

    return states_new


In [40]:
def solve_ttd_sTmT(
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, 
    *, 
    solver, f
):
    # J_full: (nτ, nt)
    # C_J: (nt,)
    # Q: (nt,)
    # ET: (nt,)
    # dt: (1,)
    # sTmT_0: (nt, 1+nm)
    # ST_top_0: (nt,)
    # ST_bot_0: (nt,)
    # sTmT_init: (nτ, 1+nm)
    # sas_Q, sas_ET: (1,)
    # α_Q, α_ET: (1,)
    # k1, C_eq: (nm,)
    
    sas_funcs = [sas_Q, sas_ET]
    other_args = [α_Q, α_ET, k1, C_eq]
    
    def step_τ(states, x):
        # sTmT_init_τ : (1+nm,)
        sTmT_init_τ, J_τ = x
        sT_init_τ = sTmT_init_τ[0]
    
        # sTmT : (nt, 1+nm)
        # ST_top, ST_bot : (nt+1,)
        sTmT, ST_top, ST_bot = states
        
        # Get fluxes and sas arguments
        fluxes = [J_τ, C_J, Q, ET]
        sas_args = [Q, ET]
    
        # Solve the ODE system
        sTmT_new = solver(
            f, sTmT, ST_top[:-1], ST_bot[1:], dt, 
            *fluxes, *sas_funcs, *sas_args, *other_args
        ) # (nt, 1+nm)
        sT_new = sTmT_new[...,0]

        # Update ST
        ST_top_new = ST_bot
        ST_bot_new = jnp.concat([sT_init_τ[None] * dt, ST_bot[1:] + sT_new * dt])
        # jax.debug.print('sT_new: {x}; ST_bot: {y}; args: {z}', x=sT_new, y=ST_bot, z=args)
        
        # Variables as the initial condition to the next step
        sTmT_new_rotate = jnp.concat([sTmT_init_τ[None,...],sTmT_new[:-1]], axis=0)
        
        return (sTmT_new_rotate, ST_top_new, ST_bot_new), sTmT_new
    
    _, sTmTs = jax.lax.scan(step_τ, (sTmT_0, ST_top_0, ST_bot_0), (sTmT_init,J_full))

    return sTmTs


In [122]:
# def calculate_cT(sT, mT):
#     y = jnp.where(sT == 0.0, 1.0, sT)
#     sT = jnp.where(sT == 0.0, 0.0, 1./ y)
#     return mT / sT
# jnp.vectorize(calculate_cT, signature='(),(nm)->(nm)')(sT_init, mT_init).shape


In [284]:
def solve_sas(
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, C_Q_0, P_old,
    *, 
    solver, f
):
    # J_full: (nτ, nt)
    # C_J: (nt, nm)
    # Q: (nt,)
    # ET: (nt,)
    # dt: (1,)
    # sTmT_0: (nt, 1+nm)
    # ST_top_0: (nt,)
    # ST_bot_0: (nt,)
    # sTmT_init: (nτ, 1+nm)
    # sas_Q, sas_ET: (1,)
    # α_Q, α_ET: (1,) 
    # k1, C_eq: (nm,)
    # C_Q_0, P_old: (nt, nm), (nt, nm)
    
    nτ, nt = J_full.shape
    
    sTmTs = solve_ttd_sTmT(
        J_full, C_J, Q, ET, dt,
        sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
        sas_Q, sas_ET, α_Q, α_ET, k1, C_eq, 
        solver=solver, f=f
    )
    
    sT = sTmTs[...,0]  # (nτ, nt)
    mT = sTmTs[...,1:]  # (nτ, nt, nm)
    
    sT_init, mT_init = sTmT_init[:,0], sTmT_init[:,1:]
    sT = jnp.concat([sT_init[:,None], sT], axis=1)  # (nτ, nt+1)
    mT = jnp.concat([mT_init[:,None,:], mT], axis=1)  # (nτ, nt+1, nm)
    
     # Calculate ST
    ST = jnp.cumsum(sT * dt, axis=0)  # (nτ, nt+1)
    
    # Calculate CT
    def calculate_cT(sT, mT):
        y = jnp.where(sT == 0.0, 1.0, sT)
        sT_inv = jnp.where(sT == 0.0, 0.0, 1./ y)
        return mT * sT_inv
    CT = jnp.vectorize(calculate_cT, signature='(),(nm)->(nm)')(
        sT[:,:nt], mT[:,:nt,:]
    )  # (nτ, nt, nm)
    
    # Calculate pQ and C_Q
    sas_Q_arg = jnp.stack([Q]*nτ)[...,None]  # (nτ, nt, 1)
    ωQ_vec = jnp.vectorize(sas_Q.pdf, signature='(),(n)->()')
    sas_Q_vec = jnp.vectorize(sas_Q, signature='(),(n)->()')
    ωQ = ωQ_vec(ST[:,1:], sas_Q_arg)
    pQ = ωQ * sT[:,1:]
    PQ = sas_Q_vec(ST[:,1:], sas_Q_arg)
    PQ_old = PQ[-1,:]  # (nt)
    # print(PQ2)
    # print(STtop[:,1:])
    # print(PQ1)
    # print(STbot[:,1:])
    print(ST[:,1:])
    print(pQ)
    
    C_Q = jnp.vectorize(lambda a,b:a*b, signature='(nm),()->(nm)')(CT, pQ)  # (nτ, nt, nm)
    C_Q = C_Q * α_Q * dt  # (nτ, nt, nm)
    C_Q = jnp.sum(C_Q, axis=0)  # (nt, nm)
    C_Q = C_Q + PQ_old[:,None] * C_old[None,:]  # (nt, nm)
    
    # Calculate pET and C_ET
    
    # m_Q = compute_mQ(sT, mT, α, Q_t, pQ) # (nτ, nt, nm)
    # part1 = α_Q * pQ / sT
    # m_Q = α_Q * mT * (α * Q_t * pQ) / sT

#     # Calculate the concentration
#     C_Q  = C_Q[-1] + C_old * P_Q_old[-1]
#     # C_ET = C_ET[-1] + C_old * P_ET_old[-1]
    
    return sT, mT, C_Q
    
#     # Calculate ST
#     STbot = jnp.cumsum(sT * dt, axis=0)  # (nτ, nt+1)
#     STtop = jnp.concat([ST_top_0[None,:], STbot[:-1,:]], axis=0)  # (nτ, nt+1)
    
#     # Calculate CT
#     def calculate_cT(sT, mT):
#         y = jnp.where(sT == 0.0, 1.0, sT)
#         sT_inv = jnp.where(sT == 0.0, 0.0, 1./ y)
#         return mT * sT_inv
#     CT = jnp.vectorize(calculate_cT, signature='(),(nm)->(nm)')(
#         sT[:,:nt], mT[:,:nt,:]
#     )  # (nτ, nt, nm)
    
#     # Calculate pQ and C_Q
#     sas_Q_arg = jnp.stack([Q]*nτ)[...,None]  # (nτ, nt, 1)
#     sas_Q_vec = jnp.vectorize(sas_Q, signature='(),(n)->()')
#     PQ1 = sas_Q_vec(STbot[:,1:], sas_Q_arg)
#     PQ2 = sas_Q_vec(STtop[:,1:], sas_Q_arg)
#     pQ = (PQ1 - PQ2) / dt  # (nτ, nt)
#     P_Q_old = PQ1[-1,:]  # (nt)
#     # print(PQ2)
#     # print(STtop[:,1:])
#     # print(PQ1)
#     # print(STbot[:,1:])
#     print(pQ)
    
#     C_Q = jnp.vectorize(lambda a,b:a*b, signature='(nm),()->(nm)')(CT, pQ)  # (nτ, nt, nm)
#     C_Q = C_Q * α_Q * dt  # (nτ, nt, nm)
#     C_Q = jnp.sum(C_Q, axis=0)  # (nt, nm)
#     C_Q = C_Q + P_Q_old[:,None] * C_old[None,:]  # (nt, nm)
    
#     # Calculate pET and C_ET
    
#     # m_Q = compute_mQ(sT, mT, α, Q_t, pQ) # (nτ, nt, nm)
#     # part1 = α_Q * pQ / sT
#     # m_Q = α_Q * mT * (α * Q_t * pQ) / sT

# #     # Calculate the concentration
# #     C_Q  = C_Q[-1] + C_old * P_Q_old[-1]
# #     # C_ET = C_ET[-1] + C_old * P_ET_old[-1]
    
#     return sT, mT, C_Q


# Run SAS

In [198]:
C_J = jnp.array(data_df[['C_J']].values) # (nt,nm)
Q = jnp.array(data_df['Q'].values) # (nt,)
ET = jnp.array(data_df['ET'].values) # (nt,)
J = jnp.array(data_df['J'].values) # (nt,)
J_full = jnp.concat([J[None,:], jnp.zeros([τ_max-1, nt])])

# Initial sT and mT at time = 0
# sT_init, mT_init = jnp.zeros(τ_max), jnp.zeros(τ_max)
sT_init, mT_init = jnp.array([10., 15., 20.]), jnp.array([[10., 10., 10.]]).T
sTmT_init = jnp.concat([sT_init[...,None], mT_init], axis=1)
nm = mT_init.shape[1]

# Initials at age = 0
sTmT_0 = jnp.zeros([nt, 1+nm])
ST_top_0, ST_bot_0 = jnp.zeros(nt+1), jnp.zeros(nt+1) 
C_Q_0, P_old = jnp.zeros([nt, nm]), jnp.ones([nt, nm]) 
# initials = [sT_0, mT_0, ST_top_0, ST_bot_0, C_Q_0, P_old]

# SAS function
# sas_func = partial(sas_gamma, a=0.02)
# sas_func = partial(sas_beta, a=2.31, b=0.627)


## Kumaraswamy

In [282]:
sas_Q = SAS_Kumaraswamy(a=2.31, b=0.627, loc=0., scale=1.)
sas_ET = SAS_null(loc=0., scale=1.)


In [257]:
# import numpy as np
# np.random.seed(3)
# a = np.random.uniform(size=(3,4))
# b = np.random.uniform(size=(3,4,1))
# sas_Q_v = jnp.vectorize(sas_Q, signature='(),(n)->()')
# sas_Q_v(a, b)

In [276]:
# sas_Q.pdf(0.99)

In [285]:
sT_final, mT_final, C_Q_final = solve_sas(
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, C_Q_0, P_old,
    solver=solve_step_rk4, f=fvec
)
sT_final, mT_final, C_Q_final


[[0.37841514 0.79494625 0.37841514 0.37841514 0.37841514]
 [3.9987285  1.1647934  0.8670014  0.64257014 0.64257014]
 [9.998729   4.7786837  1.1648097  0.9075788  0.7870171 ]]
[[0.9280047 3.1688328 0.9280047 0.9280047 0.9280047]
 [0.        0.        1.717395  1.0511175 1.0511175]
 [0.        0.        0.        0.8011572 0.5798822]]


(Array([[10.        ,  0.9460378 ,  1.9873656 ,  0.9460378 ,  0.9460378 ,
          0.9460378 ],
        [15.        ,  9.050783  ,  0.9246178 ,  1.2214657 ,  0.6603875 ,
          0.6603875 ],
        [20.        , 15.        ,  9.034726  ,  0.74452084,  0.6625216 ,
          0.36111733]], dtype=float32),
 Array([[[10.        ],
         [ 0.        ],
         [ 0.9936828 ],
         [ 0.        ],
         [ 0.        ],
         [ 0.        ]],
 
        [[10.        ],
         [ 9.050783  ],
         [ 0.        ],
         [ 0.61073285],
         [ 0.        ],
         [ 0.        ]],
 
        [[10.        ],
         [10.        ],
         [ 9.034726  ],
         [ 0.        ],
         [ 0.3312608 ],
         [ 0.        ]]], dtype=float32),
 Array([[0.37120187],
        [0.        ],
        [0.18560092],
        [0.21022351],
        [0.11597645]], dtype=float32))

In [38]:
# sT_final, mT_final, mQ_final, C_Q_final, pQ_final = solve_sas(
#     J_full, C_J, Q, J, dt,
#     sT_init, mT_init,
#     sT_0, mT_0, ST_top_0, ST_bot_0, C_Q_0, P_old,
#     sas_func,
#     α, k1, C_eq
# )
# sT_final, mT_final, mQ_final, C_Q_final, pQ_final


## MDN

In [105]:
n_input, n_hidden, n_mixture, key = 1, 20, 5, jax.random.key(42)
mlp_kwargs = {"width_size": 20, "depth":3}

sas_model = SAS_MDN(0., 1., n_input, n_hidden, n_mixture, key, **mlp_kwargs)


In [115]:
a, b = 0.5*jnp.ones([4,5]), 3+jnp.zeros([4,5,1])
jax.vmap(sas_model, in_axes=((0,1),(0,1),))(a, b)

ValueError: vmap in_axes specification must be a tree prefix of the corresponding value, got specification ((0, 1), (0, 1)) for value tree PyTreeDef((*, *)).

In [117]:
# sas_model(0.5*jnp.ones([4,5]), 3+jnp.zeros([4,5]))
# jnp.vectorize(sas_model)(0.5*jnp.ones([4,5]), 3+jnp.zeros([4,5,1]))
# jnp.vectorize(sas_model, signature='(),(n)->()')(0.5*jnp.ones([4,5]), 3+jnp.zeros([4,5,1]))


Array([[0.6779885, 0.6779885, 0.6779885, 0.6779885, 0.6779885],
       [0.6779885, 0.6779885, 0.6779885, 0.6779885, 0.6779885],
       [0.6779885, 0.6779885, 0.6779885, 0.6779885, 0.6779885],
       [0.6779885, 0.6779885, 0.6779885, 0.6779885, 0.6779885]],      dtype=float32)